# 2 - Sensitivity Test on Spread Tolerance



## Sections
1) Parameter Sweep & Trade Generation

2) Performance Aggregation & Multi-Metric Review

3) Radar Comparison Across Metrics

4) Equity Curve Comparison

5) Average Equity Benchmark


---


## 1) Parameter Sweep & Trade Generation


#### <u>Index vs. Tradable Futures</u>

Deribit options use the **BTC Index** as underlying. This index is a weighted average of spot prices across multiple exchanges ([Deribit Docs](https://support.deribit.com/hc/en-us/articles/25944739377309-Index-Prices)).  

Formally, let  

$$
I_t = \text{BTC Index at time } t
$$  

$$
F_t = \text{BTC-PERPETUAL futures price at time } t
$$  

Since the index is **not tradable**, while executions occur on futures, a structural difference (basis) arises:  

$$
\Delta_t = \frac{F_t - I_t}{I_t}
$$  

To ensure that trading signals are **economically actionable** and not artifacts of this basis,  
we introduce a **spread tolerance**:  

$$
\tau
$$  

A signal is validated only if the absolute basis remains within tolerance:  

$$
|\Delta_t| \leq \tau
$$  

This filter guarantees consistency between the theoretical option model (indexed on **BTC Index**)  
and the practical execution market (futures **BTC-PERPETUAL**).  


#### <u>Parameter Sweep & Trade Generation</u>

In this step the notebook:
- Imports the precomputed daily signals plus the 5-minute BTC futures feed.  
- Sweeps `spread_tolerance_pct` thresholds from 0.00% to 1.24% while keeping `gap_threshold` and `sl_multiplier` fixed.  
- Runs a walk-forward loop that applies the directional signal, enforces the spread gate, and simulates TP/SL/EOD exits on next-day intraday candles.  
- Collects each tolerance's trade ledger in the `results` dictionary for subsequent analysis.  

**Spread eligibility** is enforced as:  

$$
\frac{\bigl|F_{\text{entry}} - I_{\text{entry}}\bigr|}{I_{\text{entry}}} \le \tau 
$$  

**Trade P&L** is logged via:  

$$
 r =
 \begin{cases}
   \dfrac{P_{\text{exit}} - P_{\text{entry}}}{P_{\text{entry}}}, & \text{LONG} \\[6pt]
   \dfrac{P_{\text{entry}} - P_{\text{exit}}}{P_{\text{entry}}}, & \text{SHORT}
 \end{cases}
$$  


In [ ]:
# ==============================================
# BLOCK 2 - SENSITIVITY TEST ON SPREAD TOLERANCE
# ==============================================

import pandas as pd
import numpy as np
from tqdm import tqdm

# Load precomputed signals
df_signals = pd.read_csv("Data/df_signals.csv", parse_dates=["Date", "Expiry"])

# 5-minute data
df_5m = pd.read_csv("Data/btc_perpetual_5min.csv")
df_5m["timestamp"] = pd.to_datetime(df_5m["timestamp"])
df_5m = df_5m.sort_values("timestamp")
df_5m["date"] = df_5m["timestamp"].dt.date

# Fixed parameters
gap_threshold = 0.03
sl_multiplier = 2

# Parameters to test
spread_tolerance_values = np.round(np.arange(0.000, 0.0125, 0.0001), 3) 
results = {}

for spread_tolerance_pct in spread_tolerance_values:
    trades = []

    for i in tqdm(range(len(df_signals) - 1)):
        row = df_signals.iloc[i]
        today = row["Date"]
        spot = row["Spot"]
        fair_value = row["Fair_Value"]
        gap = row["Gap"]

        # Directional signal
        if gap > gap_threshold:
            direction = "LONG"
        elif gap < -gap_threshold:
            direction = "SHORT"
        else:
            continue  # No signal

        # Entry price at end of day (last 5 minutes of the day)
        df_today_5m = df_5m[df_5m["date"] == today.date()]
        if df_today_5m.empty:
            continue

        entry_row = df_today_5m.iloc[-1]
        entry_price = entry_row["close"]
        entry_time = entry_row["timestamp"]

        # Spread condition
        if abs(entry_price - spot) / spot > spread_tolerance_pct:
            continue

        # TP and SL
        fair_value_adjusted = fair_value + (entry_price - spot)
        tp = fair_value_adjusted
        sl_pct = abs(gap) * sl_multiplier
        stop = entry_price * (1 - sl_pct) if direction == "LONG" else entry_price * (1 + sl_pct)

        # Next trading day to monitor exit
        next_day = df_signals.iloc[i + 1]["Date"].date()
        df_nextday = df_5m[df_5m["date"] == next_day]
        if df_nextday.empty:
            continue

        for _, row in df_nextday.iterrows():
            high = row["high"]
            low = row["low"]
            ts = row["timestamp"]

            if direction == "LONG":
                if high >= tp:
                    exit_price, exit_time, exit_type = tp, ts, "TP"
                    break
                elif low <= stop:
                    exit_price, exit_time, exit_type = stop, ts, "SL"
                    break
            else:
                if low <= tp:
                    exit_price, exit_time, exit_type = tp, ts, "TP"
                    break
                elif high >= stop:
                    exit_price, exit_time, exit_type = stop, ts, "SL"
                    break
        else:
            # Exit at EOD
            exit_row = df_nextday.iloc[-1]
            exit_price = exit_row["close"]
            exit_time = exit_row["timestamp"]
            exit_type = "EOD"

        ret = ((exit_price - entry_price) / entry_price) if direction == "LONG" else ((entry_price - exit_price) / entry_price)

        trades.append({
            "Entry_Date": today,
            "Entry_Timestamp": entry_time,
            "Exit_Timestamp": exit_time,
            "Direction": direction,
            "Spot_Entry": entry_price,
            "Spot_Exit": exit_price,
            "Return_Pct": ret * 100,
            "Exit_Type": exit_type,
            "Fair_Value": tp,
            "Gap_Pct": gap,
            "SL_Pct": sl_pct
        })

    df_trades = pd.DataFrame(trades)
    results[spread_tolerance_pct] = df_trades

print("Sensitivity test completed.")


---


## 2) Performance Aggregation & Multi-Metric Review


#### <u>Performance Summary & Multi-Axis Visualization</u>

This block:
- Defines `compute_stats` to annualize return, Sharpe, drawdown, and hit-rate metrics for any trades DataFrame.
- Builds `df_summary` by evaluating every tolerance's ledger.
- Plots **Sharpe**, **CAGR**, and **Max Drawdown** on coordinated y-axes against the tolerance parameter.
- Surfaces the Sharpe-optimal tolerance and prints its key KPIs.

Key formulas:

$$
R_{\text{tot}} = \prod_{i=1}^{n} (1 + r_i) - 1,\qquad\text{CAGR} = (1 + R_{\text{tot}})^{1/T} - 1
$$

$$
\text{Sharpe} = \frac{\mu_r}{\sigma_r} \sqrt{N_{\text{trades per year}}}
$$

$$
\text{MaxDD} = \min_t \left( \frac{E_t - \max_{\tau \le t} E_\tau}{\max_{\tau \le t} E_\tau} \right) \times 100\%
$$


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


# Function to compute statistics for a df_trades
def compute_stats(df_trades):
    if df_trades.empty:
        return {
            "Total_Return_Pct": 0,
            "CAGR": 0,
            "Sharpe": 0,
            "Win_Rate_Pct": 0,
            "N_Trades": 0,
            "Max_Drawdown": 0
        }

    df_trades["Entry_Date"] = pd.to_datetime(df_trades["Entry_Date"])
    df_trades = df_trades.sort_values("Entry_Date").reset_index(drop=True)

    n_years = (df_trades["Entry_Date"].iloc[-1] - df_trades["Entry_Date"].iloc[0]).days / 365
    total_return = (df_trades["Return_Pct"] / 100 + 1).prod() - 1
    cagr = (1 + total_return) ** (1 / n_years) - 1 if n_years > 0 else 0
    returns = df_trades["Return_Pct"] / 100
    avg_return = df_trades["Return_Pct"].mean()
    std_return = df_trades["Return_Pct"].std()
    n_days = (df_trades["Exit_Timestamp"].max() - df_trades["Entry_Timestamp"].min()).days + 1
    trades_per_year = len(df_trades) / n_days * 365
    sharpe = avg_return / std_return * (trades_per_year ** 0.5) if std_return > 0 else 0
    win_rate = (df_trades["Return_Pct"] > 0).mean()

    # Compute equity curve and Max Drawdown
    equity = (1 + returns).cumprod()
    peak = equity.cummax()
    drawdown = (equity - peak) / peak
    max_dd = drawdown.min() * 100  # percentage

    return {
        "Total_Return_Pct": total_return * 100,
        "CAGR": cagr * 100,
        "Sharpe": sharpe,
        "Win_Rate_Pct": win_rate * 100,
        "N_Trades": len(df_trades),
        "Max_Drawdown": max_dd
    }

# Apply to every df_trades in results
summary = []
for tol, df in results.items():
    stats = compute_stats(df.copy())
    stats["spread_tolerance_pct"] = tol
    summary.append(stats)

# Convert to DataFrame
df_summary = pd.DataFrame(summary).sort_values("spread_tolerance_pct")

plt.figure(figsize=(12, 6))

# Main plot: Sharpe
fig, ax1 = plt.subplots(figsize=(12, 6))
color = 'tab:blue'
ax1.set_xlabel("Spread Tolerance (%)")
ax1.set_ylabel("Sharpe Ratio", color=color)
ax1.plot(df_summary["spread_tolerance_pct"], df_summary["Sharpe"], marker="o", color=color, label="Sharpe")
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True)

# Secondary axis: CAGR
ax2 = ax1.twinx()
color = 'tab:green'
ax2.set_ylabel("CAGR (%)", color=color)
ax2.plot(df_summary["spread_tolerance_pct"], df_summary["CAGR"], marker="s", linestyle='--', color=color, label="CAGR")
ax2.tick_params(axis='y', labelcolor=color)

# Third axis for Max DD (inverted)
ax3 = ax1.twinx()
color = 'tab:red'
ax3.spines["right"].set_position(("outward", 60))
ax3.set_ylabel("Max Drawdown (%)", color=color)
ax3.plot(df_summary["spread_tolerance_pct"], df_summary["Max_Drawdown"], marker="^", linestyle=':', color=color, label="Max DD")
ax3.tick_params(axis='y', labelcolor=color)

plt.title("Sensitivity Test – Sharpe, CAGR and Max DD vs Spread Tolerance (%)")
fig.tight_layout()
plt.show()

#===== Optimal parameter =====
best_row = df_summary.loc[df_summary["Sharpe"].idxmax()]
print(f""" Best spread_tolerance_pct: {best_row['spread_tolerance_pct']:.3f}
- Sharpe Ratio     = {best_row['Sharpe']:.2f}
- CAGR             = {best_row['CAGR']:.2f} %
- Max Drawdown     = {best_row['Max_Drawdown']:.2f} %
- N. Trades        = {best_row['N_Trades']}""")


---


## 3) Radar Comparison Across Metrics


#### <u>Normalized Multi-Metric Radar Chart</u>

Within this cell the notebook:
- Copies `df_summary` and keeps the key performance columns of interest.
- Applies **min-max scaling** (with inverted sign for Max Drawdown) to place all metrics on [0, 1].
- Generates a **radar/polar plot** so each tolerance can be visually compared across the **normalized metrics**.

Normalization rule:

$$
\tilde{x}_i = \begin{cases}\dfrac{x_i - x_{\min}}{x_{\max} - x_{\min}}, & \text{metric to maximise} \\ 1 - \dfrac{x_i - x_{\min}}{x_{\max} - x_{\min}}, & \text{metric to minimise}\end{cases}
$$


In [ ]:
from math import pi

# Select tolerances to plot
df_radar = df_summary.copy()
df_radar = df_radar[["spread_tolerance_pct", "Sharpe", "CAGR", "Max_Drawdown", "N_Trades"]]

# Normalize every column between 0 and 1 for comparison
metrics = ["Sharpe", "CAGR", "Max_Drawdown", "N_Trades"]
for col in metrics:
    if col == "Max_Drawdown":
        # Lower is better for Max DD → invert and normalize
        max_val = df_radar[col].min()
        min_val = df_radar[col].max()
        df_radar[col] = (df_radar[col] - min_val) / (max_val - min_val)
    else:
        df_radar[col] = (df_radar[col] - df_radar[col].min()) / (df_radar[col].max() - df_radar[col].min())

# Radar setup
categories = metrics
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # close the polygon

plt.figure(figsize=(10, 10))

for i, row in df_radar.iterrows():
    values = row[metrics].tolist()
    values += values[:1]  # close the polygon
    plt.polar(angles, values, marker='o', label=f'{row["spread_tolerance_pct"]:.3f}%')

plt.xticks(angles[:-1], categories, fontsize=12)
plt.title("Radar Plot – comparing metrics by Spread Tolerance", size=15, pad=20)
plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()


---


## 4) Equity Curve Comparison


#### <u>Spread-Tolerance Equity Trajectories</u>

Here the notebook:
- Iterates through the `results` dictionary and reconstructs each tolerance's **cumulative equity curve**.
- Overlays the equity paths on a single chart to assess dispersion and stability across tolerances.

Each path follows the recursion:

$$
E_t = E_{t-1} \times (1 + r_t), \qquad E_0 = 1.
$$


In [ ]:
plt.figure(figsize=(12, 6))

for tol, df in results.items():
    if df.empty:
        continue
    df = df.copy()
    df["Entry_Date"] = pd.to_datetime(df["Entry_Date"])
    df = df.sort_values("Entry_Date")
    df["Equity"] = (df["Return_Pct"] / 100 + 1).cumprod()
    label = f"{tol:.3f}%"
    plt.plot(df["Entry_Date"], df["Equity"], label=label)

plt.title("Equity Line – comparison across different Spread Tolerance values")
plt.xlabel("Date")
plt.ylabel("Equity Line (base = 1)")
plt.grid(True)
plt.legend(title="Spread Tolerance", loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()


---


## 5) Average Equity Benchmark



#### <u>Composite Equity Benchmark</u>

Finally the notebook:
- Aligns every tolerance-specific equity curve on a common date index.
- Computes the row-wise mean equity (`Equity_Media`) as an aggregate benchmark.
- Plots the composite trajectory to highlight the central tendency of the strategy family.

Aggregate equity is computed as:

$$
\bar{E}_t = \frac{1}{K} \sum_{k=1}^{K} E_t^{(k)}.
$$


In [ ]:
# Dictionary that collects all equity curves with date as index
equity_dict = {}

for tol, df in results.items():
    if df.empty:
        continue
    df = df.copy()
    df["Entry_Date"] = pd.to_datetime(df["Entry_Date"])
    df = df.sort_values("Entry_Date")
    df["Equity"] = (df["Return_Pct"] / 100 + 1).cumprod()
    equity_dict[tol] = df.set_index("Entry_Date")["Equity"]

# Merge all equity curves on the same time index
df_equity_all = pd.concat(equity_dict.values(), axis=1)

# Compute the row-wise average (ignoring NaNs)
df_equity_all["Equity_Media"] = df_equity_all.mean(axis=1)

# Plot the average
plt.figure(figsize=(12, 6))
plt.plot(df_equity_all.index, df_equity_all["Equity_Media"], color="black", linewidth=2.5, label="Average Equity")

plt.title("Average Equity Line – Spread Tolerance")
plt.xlabel("Date")
plt.ylabel("Equity Line (base = 1)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


---


## 6) Closing Notes


#### <u>Implementation Reminders</u>

- Trade generation is deterministic given the saved signals and intraday data; re-running will overwrite `results`.
- Consider stress-testing the Sharpe-optimal tolerance against alternative TP/SL multipliers before live deployment.
- Equity plots assume continuous compounding without transaction costs; incorporate execution frictions for production sizing.

Net-of-cost check:

$$
r_{\text{net}} = r_{\text{gross}} - c_{\text{frictions}}
$$
